In [ ]:
import tifffile as tf 
from pathlib import Path
import skimage
from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt
import numpy as np
import cv2
import pickle 
from tqdm import tqdm 
import seaborn as sns 
import pandas as pd
from sklearn.metrics import f1_score
from tqdm import tqdm
from skimage.measure import regionprops
import matplotlib as mpl 

In [ ]:
keys = [ ] #sample identifiers

In [ ]:
keys_name = {  } #sample identifier : sample name for plotting 

In [ ]:
registration_path = r'' # path for registered images

In [ ]:
#Load in registered images
#Compute SSIM between image pairs 

ssim_score = {}
dice_score = {}

for key in tqdm(keys):
    
    key_registration_path = registration_path + key
    dapi_paths = [f for f in Path(key_registration_path).rglob('*.tif') if 'registered' in str(f) and 'CH1' in str(f)]
    color_paths =  [f for f in Path(key_registration_path).rglob('*.tif') if 'registered' in str(f) and 'Cycle' not in str(f)]
    channels = {}

    for channel in dapi_paths:
        channels[str(channel).split('\\')[-1].split('.')[0].split('_')[0]] = tf.imread(channel)

    for channel in color_paths:
        channels[str(channel).split('\\')[-1].split('.')[0].split('_')[0]] = tf.imread(channel)
    ssim_score[keys_name[key]] = []


    i = 0
    for channel_name, image in channels.items():
        print(channel_name)
        if(i == 0):
            i+=1
            continue
        if('Cycle' in channel_name):
            reference = list(channels.values())[0].copy().astype(np.float32)
            reference[reference <= skimage.filters.threshold_otsu(reference)] = 0
            image_to_register = image.copy().astype(np.float32)
            image_to_register[image_to_register <= skimage.filters.threshold_otsu(image_to_register)] = 0

            reference = cv2.normalize(reference, None, 0, 1, cv2.NORM_MINMAX)
            image_to_register = cv2.normalize(image_to_register, None, 0, 1, cv2.NORM_MINMAX)
            data_range = max(
                image_to_register.max() - image_to_register.min(),
                reference.max() - reference.min()
            )
            ssim_score[keys_name[key]].append(ssim(reference, image_to_register, data_range=data_range))

        else:
            if('H&E' in channel_name):
                reference = list(channels.values())[0].copy().astype(np.float32)
                reference[reference <= skimage.filters.threshold_otsu(reference)] = 0
                reference = cv2.normalize(reference, None, 0, 1, cv2.NORM_MINMAX)
            else:
                if('H&E' in list(channels.keys())): 
                    reference =  skimage.color.rgb2gray(channels['H&E'].copy()).astype(np.float32)
                else:
                    reference =  skimage.color.rgb2gray(channels['HE'].copy()).astype(np.float32)
            
            image_gray = skimage.color.rgb2gray(image).astype(np.float32)
            image_edges =  cv2.normalize(cv2.Laplacian(image_gray, cv2.CV_32F), None, 0, 1, cv2.NORM_MINMAX)

            reference_edges = cv2.normalize(cv2.Laplacian(reference, cv2.CV_32F), None, 0, 1, cv2.NORM_MINMAX)

            data_range = max(
                image_edges.max() - image_edges.min(),
                reference_edges.max() - reference_edges.min()
            )

            ssim_score[keys_name[key]].append(ssim(image_edges, reference_edges, data_range=data_range))


In [ ]:
#sample identifier : um / pixel 
um_per_pixel_dict = {
}

In [ ]:
segmentation_masks = {}
for key in tqdm(keys):
    key_registration_path = registration_path + key
    segmentation_mask_paths = [f for f in Path(key_registration_path).rglob('*.pickle') if 'segmentation_mask' in str(f)]
    segmentation_masks[key] = {}

    for mask in segmentation_mask_paths:
        with open(mask, 'rb') as f:
            segmentation_masks[key][str(mask).split('\\')[-1].split('.')[0].split('_')[-1]] = pickle.load(f)


In [ ]:
def match_cells_by_overlap(
    mask_cycle_1,
    mask_cycle_2,
):
    mask1 = np.asarray(mask_cycle_1, dtype=np.int64)
    mask2 = np.asarray(mask_cycle_2, dtype=np.int64)

    labels1, areas1 = np.unique(mask1, return_counts=True)
    labels2, areas2 = np.unique(mask2, return_counts=True)

    area_map1 = dict(zip(labels1, areas1))
    area_map2 = dict(zip(labels2, areas2))

    max_label2 = int(mask2.max()) + 1

    pair_id = mask1 * max_label2 + mask2

    valid_pixels = (mask1 > 0) & (mask2 > 0)
    pair_id = pair_id[valid_pixels]

    unique_pairs, overlap_pixels = np.unique(
        pair_id,
        return_counts=True,
    )

    labels_pair1 = unique_pairs // max_label2
    labels_pair2 = unique_pairs % max_label2

    ious = np.array([
        overlap / (
            area_map1[label1]
            + area_map2[label2]
            - overlap
        )
        for label1, label2, overlap in zip(
            labels_pair1,
            labels_pair2,
            overlap_pixels,
        )
    ])

    candidate_indices = np.where(ious >0)[0]

    candidate_indices = candidate_indices[
        np.argsort(ious[candidate_indices])[::-1]
    ]

    matches = {}
    match_ious = {}

    assigned_cycle_1 = set()
    assigned_cycle_2 = set()

    for i in candidate_indices:
        label1 = int(labels_pair1[i])
        label2 = int(labels_pair2[i])

        if label1 in assigned_cycle_1:
            continue

        if label2 in assigned_cycle_2:
            continue

        matches[label1] = label2
        match_ious[label1] = float(ious[i])

        assigned_cycle_1.add(label1)
        assigned_cycle_2.add(label2)

    return matches




In [ ]:

def compute_registration_error_um(mask_cycle_1, mask_cycle_2, cell_matches, pixel_size_um):

    props_1 = {prop.label: prop.centroid for prop in regionprops(mask_cycle_1)}
    props_2 = {prop.label: prop.centroid for prop in regionprops(mask_cycle_2)}
    
    errors_px = []
    
    for label_1, label_2 in cell_matches.items():
        if label_1 in props_1 and label_2 in props_2:
            centroid_1 = np.array(props_1[label_1])
            centroid_2 = np.array(props_2[label_2])
            
            # Distance formula in pixel units
            distance_px = np.sqrt(np.sum((centroid_1 - centroid_2) ** 2))
            errors_px.append(distance_px)
            
    errors_px = np.array(errors_px)

    errors_um = errors_px * pixel_size_um

    
    return errors_um


In [ ]:
registration_errors = {}
median_registration_error = {}

In [ ]:

for key, masks in tqdm(segmentation_masks.items()):
    median_registration_error[key] = {}
    registration_errors[key] = {}
    reference = list(segmentation_masks[key].values())[0]
    for cycle, mask in masks.items():
        if(cycle == list(segmentation_masks[key].keys())[0]):
            continue 
        else:
            matched_cells = match_cells_by_overlap(reference, mask)
            registration_errors[key][cycle] = compute_registration_error_um(reference, mask, matched_cells, um_per_pixel_dict[key])
            median_registration_error[key][cycle] = np.median(registration_errors[key][cycle])


In [ ]:
mpl.rcParams["font.size"] = 25

data_list = []
for sample, cycles_dict in registration_errors.items():
    display_name = keys_name.get(sample, sample)
    for cycle, errors in cycles_dict.items():
        for error in errors:
            data_list.append({
                "Sample": display_name,
                "Cycle": cycle,
                "Registration Error (um)": error
            })

df = pd.DataFrame(data_list)

unique_cycles = list(dict.fromkeys(
    cycle for sample in registration_errors.values() for cycle in sample.keys()
))
palette = dict(zip(unique_cycles, plt.cm.tab10(np.linspace(0, 1, len(unique_cycles)))))

fig, ax = plt.subplots(figsize=(14, 8))

sns.boxplot(
    data=df,
    x="Sample",
    y="Registration Error (um)",
    hue="Cycle",
    hue_order=unique_cycles, 
    palette=palette,   
    linecolor="black",       
    linewidth=1,
    ax=ax,
    showfliers=False
)


for collection in ax.collections:
    collection.set_alpha(0.5)
    collection.set_edgecolor("black")
    collection.set_linewidth(1)


ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

sns.move_legend(ax, "upper left", bbox_to_anchor=(1.05, 1.0))

plt.tight_layout()
plt.show()


In [ ]:
dice = {}

for key, masks in segmentation_masks.items():
    print(key)
    dice[key] = {}
    reference = list(segmentation_masks[key].values())[0] > 0
    for cycle, mask in masks.items():
        if(cycle == list(segmentation_masks[key].keys())[0]):
            continue 
        else:
            mask_binary = mask > 0
            dice[key][cycle] = f1_score(reference.ravel(), mask_binary.ravel(), zero_division=0)


In [ ]:

mpl.rcParams["font.size"] = 25

dice_score_all = []
for sample, quality_scores in dice.items():
    for cycle, score in quality_scores.items():
        dice_score_all.append({'Sample': sample.split('/')[-1], 'Image Quality': score})  #.split('/')[-1]

dice_score_all_df = pd.DataFrame(dice_score_all)
plt.figure(figsize=(12, 6))

ax = sns.boxplot(data=dice_score_all_df, x='Sample', y='Image Quality', palette='tab20')

ax.set_xticklabels([keys_name[s] for s in list(dice.keys())],rotation=45)

sns.stripplot(data=dice_score_all_df, x='Sample', y='Image Quality', jitter=True, color='black', alpha=0.6)

plt.xlabel("Sample")
plt.ylabel("Dice Coefficient")
plt.ylim(0, 1)

plt.show()


In [ ]:
with open(registration_path + '/registration_dice_if.pickle', 'wb') as f: 
    pickle.dump(dice, f)

In [ ]:
with open(registration_path + '/registration_error_if.pickle', 'wb') as f: 
    pickle.dump(registration_errors, f)

In [ ]:
with open(registration_path + '/registration_ssim.pickle', 'wb') as f: 
    pickle.dump(ssim_score, f)

In [ ]:
with open(registration_path + '/registration_dice_if.pickle', 'rb') as f: 
    dice = pickle.load(f)

In [ ]:
with open(registration_path + '/registration_error_if.pickle', 'rb') as f: 
    registration_errors = pickle.load(f)

In [ ]:
with open(registration_path + '/registration_ssim.pickle', 'rb') as f: 
    ssim_score = pickle.load(f)

In [ ]:
for sample, cycle_error_dict in registration_errors.items():
    print(sample)
    for cycle, error in cycle_error_dict.items():
        print(cycle)
        print(np.median(error))
        print(np.mean(error))

In [ ]:

mpl.rcParams["font.size"] = 25

ssim_score_all = []
for sample, quality_scores in ssim_score.items():
    for score in quality_scores:
        ssim_score_all.append({'Sample': sample, 'Image Quality': score})  #.split('/')[-1]

ssim_score_all_df = pd.DataFrame(ssim_score_all)
plt.figure(figsize=(12, 6))

ax = sns.boxplot(data=ssim_score_all_df, x='Sample', y='Image Quality', palette='tab20')

ax.set_xticklabels(ax.get_xticklabels(),rotation=45)

sns.stripplot(data=ssim_score_all_df, x='Sample', y='Image Quality', jitter=True, color='black', alpha=0.6)

plt.xlabel("Sample")
plt.ylabel("SSIM")
plt.ylim(0, 1)

plt.show()
